In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow import keras

import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [5]:
fashion_mnist = keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

In [6]:
x_train.shape,x_test.shape

((60000, 28, 28), (10000, 28, 28))

In [7]:
x_train=x_train.reshape(60000,-1)
x_test=x_test.reshape(10000,-1)

In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler=MinMaxScaler()

x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [9]:
x_train=x_train.reshape(-1,28,28,1)
x_test=x_test.reshape(-1,28,28,1)

In [10]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

y_train=le.fit_transform(y_train)
y_test=le.fit_transform(y_test)

In [11]:
x_train.shape,y_train.shape

((60000, 28, 28, 1), (60000,))

In [12]:
class Mynn(nn.Module):
  def __init__(self):
    super().__init__()

    self.model =nn.Sequential(
        nn.Conv2d(1,32,kernel_size=3,padding="same"),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),
        nn.Conv2d(32,64,kernel_size=3,padding="same"),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),
        nn.Flatten(),
        nn.Linear(3136,128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64,10)
    )
  def forward(self,x):
    x=x.permute(0,3,1,2)
    return self.model(x)

In [18]:
learning_rate=0.1
epochs=1

In [19]:
model=Mynn()

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=learning_rate)

In [21]:
from torch.utils.data import TensorDataset, DataLoader

x_train_t = torch.tensor(x_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

train_dataset = TensorDataset(x_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [22]:
for epoch in range(epochs):
  model.train()

  for x_batch,y_batch in train_loader:
    outputs=model(x_batch)
    loss=criterion(outputs,y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [23]:
from torch.utils.data import TensorDataset, DataLoader

x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

test_dataset = TensorDataset(x_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [24]:
model.eval()

Mynn(
  (model): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=128, bias=True)
    (8): ReLU()
    (9): Dropout(p=0.3, inplace=False)
    (10): Linear(in_features=128, out_features=64, bias=True)
    (11): ReLU()
    (12): Dropout(p=0.3, inplace=False)
    (13): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [25]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
  for x_batch,y_batch in test_loader:
    output = model(x_batch)
    _, predicted = torch.max(output.data, 1)
    total += y_batch.size(0)
    correct += (predicted == y_batch).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the model on the test images: {accuracy:.2f}%')

Accuracy of the model on the test images: 10.00%
